# Lab 13: Web Search pour Modèles SOTA (MLE-STAR Component)

**Navigation** : [Lab 12 <<](../Day5-DS-Star/Lab12-DS-Star-Workshop.ipynb) | [Index](../../README.md) | [>> Lab 14](Lab14-Ablation-Refinement.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter la recherche web pour trouver les modèles SOTA
2. Extraire des informations depuis arXiv et autres sources académiques
3. Générer du code initial basé sur les résultats de recherche
4. Intégrer la recherche dans un pipeline ML automatisé

### Prérequis
- Lab 12 (DS-STAR Workshop) complété
- Compréhension des architectures ML
- Configuration multi-provider active

### Durée estimée : 35-45 minutes

> **Repères bibliographiques.** Ce lab implémente le composant **recherche web (search)** de l'agent MLE-STAR — un agent d'ingénierie ML de Google Research (arXiv:2506.15692, *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted Refinement*, 2025) qui, plutôt que de s'appuyer sur la connaissance paramétrique du LLM, recherche activement les modèles SOTA (arXiv, leaderboards) avant de générer du code. Ce principe — *augmenter* le LLM par récupération externe de connaissances — est le paradigme **RAG** (Retrieval-Augmented Generation) formalisé par P. Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, arXiv:2005.11401, NeurIPS 2020.

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import requests
from typing import List, Dict, Optional
from dataclasses import dataclass
from datetime import datetime

from config import get_settings
from utils import LLMClient

print("Imports OK : json, re, requests, dataclasses, config, utils")

Imports OK : json, re, requests, dataclasses, config, utils


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. Data Classes

In [1]:
@dataclass
class SearchResult:
    title: str
    url: str
    snippet: str
    source: str
    date: str = None

@dataclass
class ModelInfo:
    name: str
    paper_url: str
    description: str
    code_url: str = None
    performance: str = None

print("Dataclasses definies : SearchResult (titre, url, snippet, source), ModelInfo (nom, paper, description, code_url, performance)")

Dataclasses definies : SearchResult (titre, url, snippet, source), ModelInfo (nom, paper, description, code_url, performance)


## 3. Web Search Module

Ce module incarne l'étape de **retrieval** du paradigme RAG (Lewis et al. 2020) : l'agent interroge une source externe (web, arXiv) pour obtenir un contexte fraîchement récupéré, puis le passe au LLM qui génère du code fondé sur l'état de l'art observé plutôt que sur sa mémoire paramétrique seule.

In [1]:
class WebSearcher:
    """Recherche web pour trouver des modeles SOTA."""

    def __init__(self):
        self.headers = {'User-Agent': 'Mozilla/5.0 (compatible; DS-Star/1.0)'}

    def search_arxiv(self, query: str, max_results: int = 5) -> List[SearchResult]:
        """Recherche sur arXiv via API."""
        url = f"http://export.arxiv.org/api/query?search_query=all:{query}&max_results={max_results}"
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            results = []
            # Parse Atom feed
            entries = response.text.split('<entry>')[1:]
            for entry in entries:
                title = re.search(r'<title>(.*?)</title>', entry, re.DOTALL)
                link = re.search(r'<id>(.*?)</id>', entry)
                summary = re.search(r'<summary>(.*?)</summary>', entry, re.DOTALL)
                if title and link:
                    results.append(SearchResult(
                        title=title.group(1).strip().replace('\n', ' '),
                        url=link.group(1).strip(),
                        snippet=summary.group(1).strip()[:200] if summary else '',
                        source='arxiv'
                    ))
            return results
        except Exception as e:
            print(f"Erreur arXiv: {e}")
            return []

    def search_papers_with_code(self, task: str) -> List[SearchResult]:
        """Simule une recherche sur Papers With Code (API publique)."""
        # Papers With Code a une API publique mais limitee
        # Pour ce lab, on simule avec des resultats types
        mock_results = [
            SearchResult(
                title="State-of-the-Art Image Classification",
                url="https://paperswithcode.com/sota/image-classification-on-imagenet",
                snippet="Vision Transformers achieve 90.5% top-1 accuracy",
                source="paperswithcode"
            ),
            SearchResult(
                title="Object Detection Benchmarks",
                url="https://paperswithcode.com/sota/object-detection-on-coco",
                snippet="YOLOv9 and DINO lead COCO detection",
                source="paperswithcode"
            )
        ]
        return mock_results

print("Classe WebSearcher definie : recherche arXiv (API Atom) et Papers With Code (simule)")

Classe WebSearcher definie : recherche arXiv (API Atom) et Papers With Code (simule)


## 4. LLM-Based Model Extractor

In [1]:
class ModelExtractor:
    """Extrait les informations de modeles depuis les resultats de recherche."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def extract_models(self, search_results: List[SearchResult], task: str) -> List[ModelInfo]:
        """Utilise le LLM pour extraire les modeles pertinents."""
        context = "\n".join([
            f"- {r.title}: {r.snippet} ({r.url})"
            for r in search_results[:5]
        ])

        prompt = f"""Analyse ces resultats de recherche pour la tache: {task}

RESULTATS:
{context}

Extrait les 2-3 modeles les plus pertinents avec leurs informations.
Format JSON:
[
  {{"name": "...", "paper_url": "...", "description": "...", "code_url": "..."}}
]

JSON:"""

        response = self.llm.generate(prompt, temperature=0.2)

        # Extract JSON
        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                return [ModelInfo(**m) for m in data]
        except:
            pass
        return []

print("Classe ModelExtractor definie : extraction de modeles pertinents depuis resultats de recherche via LLM")

Classe ModelExtractor definie : extraction de modeles pertinents depuis resultats de recherche via LLM


## 5. Code Generator from SOTA

In [1]:
class SOTACodeGenerator:
    """Genere du code initial base sur les modeles SOTA trouves."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_initial_code(self, task: str, models: List[ModelInfo]) -> str:
        """Genere du code de base utilisant les modeles SOTA."""
        models_desc = "\n".join([
            f"- {m.name}: {m.description}"
            for m in models[:2]
        ])

        prompt = f"""Genere du code Python initial pour cette tache ML.

TACHE: {task}

MODELES SOTA DISPONIBLES:
{models_desc}

Genere un script Python simple et commente qui:
1. Charge les donnees
2. Prepare le modele
3. Entraîne et evalue

```python
# Ton code ici
```"""

        response = self.llm.generate(prompt, temperature=0.3)
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else response

print("Classe SOTACodeGenerator definie : generation de code Python initial base sur modeles SOTA")

Classe SOTACodeGenerator definie : generation de code Python initial base sur modeles SOTA


## 6. MLE-STAR Pipeline Partiel

In [1]:
class MLEStarSearcher:
    """Pipeline de recherche SOTA (partie de MLE-STAR)."""

    def __init__(self):
        self.llm = LLMClient()
        self.web_searcher = WebSearcher()
        self.extractor = ModelExtractor(self.llm)
        self.code_gen = SOTACodeGenerator(self.llm)

    def find_sota_models(self, task: str) -> Dict:
        """Trouve les modeles SOTA pour une tache donnee."""
        print(f"[WEB SEARCH] Recherche pour: {task}")

        # Search arXiv
        print("  - Recherche arXiv...")
        arxiv_results = self.web_searcher.search_arxiv(task)

        # Search Papers With Code (simule)
        print("  - Recherche Papers With Code...")
        pwc_results = self.web_searcher.search_papers_with_code(task)

        all_results = arxiv_results + pwc_results
        print(f"  - {len(all_results)} resultats trouves")

        # Extract models with LLM
        print("[EXTRACTOR] Extraction des modeles...")
        models = self.extractor.extract_models(all_results, task)
        print(f"  - {len(models)} modeles identifies")

        return {
            'task': task,
            'search_results': all_results,
            'models': models
        }

    def generate_baseline_code(self, task: str, models: List[ModelInfo]) -> str:
        """Genere du code de base."""
        print("[CODE GEN] Generation du code initial...")
        code = self.code_gen.generate_initial_code(task, models)
        return code

print("Classe MLEStarSearcher definie : pipeline complet WebSearch -> ExtractModel -> GenerateCode")

  - Recherche arXiv...


## 7. Test du Pipeline

In [8]:
# Test avec une tache ML typique
searcher = MLEStarSearcher()

task = "image classification imagenet"
result = searcher.find_sota_models(task)

print("\n" + "="*50)
print("MODELES TROUVES:")
print("="*50)
for m in result['models']:
    print(f"\n- {m.name}")
    print(f"  Description: {m.description[:80]}...")
    print(f"  Paper: {m.paper_url}")

[WEB SEARCH] Recherche pour: image classification imagenet
  - Recherche arXiv...


  - Recherche Papers With Code...
  - 7 resultats trouves
[EXTRACTOR] Extraction des modeles...



  - 3 modeles identifies

MODELES TROUVES:

- VIPriors Image Classification Challenge Model
  Description: Un modèle développé pour le défi VIPriors Image Classification, une tâche de cla...
  Paper: http://arxiv.org/abs/2007.08722v1

- HistoGAN
  Description: Un modèle génératif utilisé pour l'augmentation synthétique sélective des donnée...
  Paper: http://arxiv.org/abs/2111.06399v1

- Deep Learning Ensemble Models
  Description: Une approche utilisant des modèles d'ensemble d'apprentissage profond spécifique...
  Paper: http://arxiv.org/abs/2102.00515v3
Provider List: https://docs.litellm.ai/docs/providers



## 8. Generation de Code

In [9]:
# Generer du code initial
if result['models']:
    code = searcher.generate_baseline_code(task, result['models'])
    print("\n" + "="*50)
    print("CODE GENERE:")
    print("="*50)
    print(code[:800] + "..." if len(code) > 800 else code)

[CODE GEN] Generation du code initial...



Provider List: https://docs.litellm.ai/docs/providers


CODE GENERE:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50
import os

# Configuration de l'appareil (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation de l'appareil : {device}")

# ==========================================
# 1. CHARGER LES DONNÉES (ImageNet)
# ==========================================
# Note : ImageNet est un jeu de données massif. Vous devez l'avoir téléchargé 
# et organisé dans des dossiers 'train' et 'val'.
data_dir = './chemin_vers_imagenet' # À REMPLACER par votre chemin réel

# Définition des transformations (Augmentation de données type VIPriors/Standard)
transform_train = transforms.Compo...


## 9. Resume du Lab
### Ce que nous avons implemente
1. **WebSearcher**: Recherche sur arXiv et Papers With Code
2. **ModelExtractor**: Extraction des modeles via LLM
3. **SOTACodeGenerator**: Generation de code initial
### Limitations
- L'API Papers With Code est simulee (rate limiting)
- La recherche web reelle necessite une API (Tavily, Serper)
- Le code genere est un point de depart, pas une solution complete
### Integration MLE-STAR
Dans MLE-STAR complet, ce module est utilise pour:
1. Comprendre la competition Kaggle
2. Trouver les approches SOTA
3. Generer le code initial
4. Puis passer a l'ablation et raffinement
### Prochaine etape
- **Lab 14**: Ablation et Raffinement Cible

## Exercice

À vous de rechercher des modèles SOTA pour une autre tâche ML et de comparer les résultats !


In [10]:
# Exercice : Recherchez des modeles SOTA pour une autre tache
# Utilisez le MLEStarSearcher pour trouver des modeles pour "object detection"

# TODO: Definissez la tache de recherche
task_detection = "object detection coco"

print("=" * 50)
print(f"Recherche SOTA pour : {task_detection}")
print("=" * 50)

# TODO: Utilisez searcher.find_sota_models() pour trouver les modeles
# Indice: meme pattern que le test de la section 7
result_detection = None  # Remplacez None

# TODO: Affichez les modeles trouves
# Indice: parcourez result_detection['models'] et affichez name, description, code_url

# TODO: Comparez avec les resultats de classification d'images
# Indice: comparez len(result['models']) vs len(result_detection['models'])

# REFLEXION (repondez dans un commentaire):
# Quelle est la difference principale entre les modeles
# de classification d'images et ceux de detection d'objets ?
# ...


Recherche SOTA pour : object detection coco


## Exercice : Comparaison de Sources Academiques

Creez une fonction qui compare les resultats de recherche provenant de sources differentes (arXiv vs. Papers With Code) et identifie les modeles qui apparaissent dans les deux sources.

### Objectifs
1. Executer une recherche sur arXiv et sur Papers With Code pour la meme tache
2. Identifier les modeles communs aux deux sources (intersection)
3. Classer les modeles par nombre d'occurrences et pertinence

**Indice :**
- Utilisez `web_searcher.search_arxiv(task)` et `web_searcher.search_papers_with_code(task)`
- Comparez les titres en utilisant la similarite de chaines (`difflib.SequenceMatcher`)
- Les modeles presents dans les deux sources sont generalement plus fiables

In [11]:
# Exercice : Comparaison multi-source pour validation croisee
# Objectif : Identifier les modeles confirmes par plusieurs sources

import difflib

def compare_sources(web_searcher, task: str, similarity_threshold: float = 0.6) -> dict:
    """
    Compare les resultats arXiv et Papers With Code pour une meme tache.
    
    Args:
        web_searcher: instance de WebSearcher
        task: tache ML a rechercher
        similarity_threshold: seuil de similarite pour considerer deux resultats comme identiques
        
    Returns:
        Dictionnaire avec resultats par source, intersection et classement
    """
    # Etape 1: Recherchez sur les deux sources
    # arxiv_results = web_searcher.search_arxiv(task)
    # pwc_results = web_searcher.search_papers_with_code(task)
    
    # Etape 2: Pour chaque paire de resultats, calculez la similarite
    # Indice: difflib.SequenceMatcher(None, titre1, titre2).ratio()
    cross_source_matches = []
    
    # TODO etudiant : implentez la double boucle de comparaison
    # for ar in arxiv_results:
    #     for pwc in pwc_results:
    #         similarity = difflib.SequenceMatcher(None, ar.title.lower(), pwc.title.lower()).ratio()
    #         if similarity >= similarity_threshold:
    #             cross_source_matches.append({...})
    
    # Etape 3: Classez les resultats par pertinence
    # Indice: les matches cross-source sont plus fiables que les single-source
    
    return {
        'task': task,
        'cross_source_matches': cross_source_matches,
        'total_arxiv': 0,  # TODO
        'total_pwc': 0,     # TODO
    }

# TODO: Testez la comparaison
# searcher_web = WebSearcher()
# comparison = compare_sources(searcher_web, "image segmentation")
# print(f"Tache: {comparison['task']}")
# print(f"Resultats arXiv: {comparison['total_arxiv']}")
# print(f"Resultats PwC: {comparison['total_pwc']}")
# print(f"Matches cross-source: {len(comparison['cross_source_matches'])}")

print("Exercice a completer : comparaison multi-source pour validation croisee")

Exercice a completer


## Exercice : Evaluation de la Qualite du Code Genere

Le `SOTACodeGenerator` produit du code initial a partir des modeles SOTA trouves. L'objectif est de creer un evaluateur automatique qui verifie la qualite de ce code avant de l'utiliser.

### Objectifs
1. Implementer des verifications statiques (imports valides, variables definies, pas de syntax errors)
2. Verifier la presence d'etapes essentielles (chargement, preprocessing, entrainement, evaluation)
3. Generer un score de completude du code

**Indice :**
- `ast.parse(code)` pour verifier la syntaxe Python sans executer
- Cherchez des mots-cles : `import`, `fit`, `predict`, `score`, `print`
- Un code ML complet devrait contenir au minimum : chargement de donnees + entrainement + evaluation

In [12]:
# Exercice : Evaluateur statique de code ML genere
# Objectif : Verifier la qualite du code avant execution

import ast

class CodeQualityEvaluator:
    """Evaluateur statique de code ML genere par le LLM."""
    
    # Etapes essentielles d'un pipeline ML
    REQUIRED_STEPS = {
        'data_loading': ['read_csv', 'load', 'read_json', 'DataFrame'],
        'preprocessing': ['fillna', 'dropna', 'transform', 'fit_transform', 'scale', 'encode'],
        'training': ['fit(', 'train', 'compile'],
        'evaluation': ['score', 'accuracy', 'predict', 'evaluate', 'cross_val']
    }
    
    def evaluate_syntax(self, code: str) -> dict:
        """Verifie la syntaxe Python du code."""
        try:
            ast.parse(code)
            return {'valid': True, 'error': None}
        except SyntaxError as e:
            return {'valid': False, 'error': str(e)}
    
    def evaluate_completeness(self, code: str) -> dict:
        """
        Evalue la completude du code ML.
        
        Returns:
            Dictionnaire avec score par etape et score global
        """
        code_lower = code.lower()
        scores = {}
        
        # TODO etudiant : pour chaque etape dans REQUIRED_STEPS,
        # verifiez si au moins un mot-cle est present dans le code
        for step, keywords in self.REQUIRED_STEPS.items():
            found = any(kw in code_lower for kw in keywords)
            scores[step] = 1.0 if found else 0.0
        
        # Score global
        scores['global'] = sum(scores.values()) / len(self.REQUIRED_STEPS)
        
        return scores
    
    def evaluate_imports(self, code: str) -> list:
        """Liste les imports detectes dans le code."""
        imports = []
        for line in code.split('\n'):
            line = line.strip()
            if line.startswith('import ') or line.startswith('from '):
                imports.append(line)
        return imports

# TODO: Testez l'evaluateur sur le code genere precedemment
# evaluator = CodeQualityEvaluator()
# 
# # Utilisez le code genere dans la section precedente
# # code_genere = result['code'] si disponible, ou un exemple
# test_code = """
# import pandas as pd
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# df = pd.read_csv('data.csv')
# model = RandomForestClassifier()
# model.fit(X_train, y_train)
# score = accuracy_score(y_test, model.predict(X_test))
# print(f"Accuracy: {score}")
# """
# 
# syntax = evaluator.evaluate_syntax(test_code)
# completeness = evaluator.evaluate_completeness(test_code)
# imports = evaluator.evaluate_imports(test_code)
# 
# print(f"Syntaxe valide: {syntax['valid']}")
# print(f"Completude: {completeness}")
# print(f"Imports: {imports}")

print("Exercice a completer : evaluation statique de la qualite du code ML genere")

Exercice a completer


## Références

1. P. Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, arXiv:2005.11401, NeurIPS 2020. Paradigme RAG : augmenter un LLM par récupération externe de connaissances avant génération — fondement du module Web Search de ce lab.
2. Google Research, *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted Refinement*, arXiv:2506.15692, 2025. Agent MLE dont ce lab implémente le composant recherche SOTA (combinaison recherche + raffinement ciblé).
3. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2025. Cadre conceptuel des agents LLM (suite Labs 8-12).